In [ ]:
# ── Cell 0: Colab Bootstrap ─────────────────────────────────────────────
# Run ONCE on fresh Colab runtime. Installs deps then restarts kernel.
# After restart, skip this cell and run from Cell 1 onward.

import sys, os

if 'google.colab' in sys.modules:
    print('Installing Colab dependencies...')
    %pip install --upgrade --no-cache-dir unsloth unsloth_zoo gradio
    os._exit(00)  # Restart kernel
else:
    print('Local environment detected \u2014 skipping Colab installs.')

Installing Colab dependencies...


In [1]:
# ── Cell 1: Imports + Google Drive Mount ───────────────────────────

import sys
import os
import json
import uuid
import torch
from pathlib import Path
from datetime import datetime, timezone

IS_COLAB = 'google.colab' in sys.modules

# Google Drive mount for persistent session storage
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/subscriber-sim/data')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Google Drive mounted. Data dir: {DATA_DIR}')
else:
    DATA_DIR = Path('data')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Local mode. Data dir: {DATA_DIR}')

# Per-archetype training data files (e.g. data/horny.jsonl, data/cold.jsonl)
ARCHETYPE_NAMES = ['horny', 'cheapskate', 'casual', 'troll', 'whale', 'cold', 'simp']

def archetype_file(name):
    return DATA_DIR / f'{name}.jsonl'

print(f'Training data dir: {DATA_DIR}')
print(f'Archetype files: {", ".join(f"{n}.jsonl" for n in ARCHETYPE_NAMES)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted. Data dir: /content/drive/MyDrive/subscriber-sim/data
Training data dir: /content/drive/MyDrive/subscriber-sim/data
Archetype files: horny.jsonl, cheapskate.jsonl, casual.jsonl, troll.jsonl, whale.jsonl, cold.jsonl, simp.jsonl


In [2]:
# ── Cell 2: Load Llama 3.1 8B via Unsloth ──────────────────────────
#
# TRAINING_MODE = True  → skips for_inference(), run Cells 6-9 to fine-tune
# TRAINING_MODE = False → loads model for inference (data collection / chat)

TRAINING_MODE  = False   # ← flip to False after training

MODEL_NAME     = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 1024   # reduced from 2048 to fit T4 VRAM
DTYPE          = None
LOAD_IN_4BIT   = True

if IS_COLAB:
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = DTYPE,
        load_in_4bit   = LOAD_IN_4BIT,
    )
    tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')

    if not TRAINING_MODE:
        FastLanguageModel.for_inference(model)
        print(f'Model loaded for INFERENCE: {MODEL_NAME}')
    else:
        print(f'Model loaded for TRAINING: {MODEL_NAME}')
        print('Run Cells 6-9 to fine-tune, then set TRAINING_MODE=False.')
else:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = None
    FastLanguageModel = None
    print(f'Tokenizer-only mode (local): {MODEL_NAME}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


Model loaded for INFERENCE: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit


In [3]:
# ── Cell 3: Subscriber Archetype Definitions ───────────────────────

ARCHETYPES = {
    'horny': {
        'label': '\U0001f525 Horny',
        'system': """You are a sexually forward OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're extremely turned on and direct about what you want
- You ask about explicit content, nudes, custom videos
- You're willing to pay for content but want to be teased first
- You use explicit language and sexual emojis \U0001f346\U0001f4a6\U0001f525\U0001f60d
- You compliment her body, especially her dick/ass/tits
- You ask for sexting, JOI, custom content
- You respond eagerly to any sexual teasing
- Keep messages 1-3 sentences, casual texting style
- You're a guy who's into trans women and not shy about it

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey sexy \U0001f60f saw ur page and damn... u got me hard already',
    },

    'cheapskate': {
        'label': '\U0001f4b8 Cheapskate',
        'system': """You are a cheap OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're interested in her content but ALWAYS negotiate the price down
- You say things like "that's too much", "can I get a discount?", "what about half price?"
- You claim other creators charge less
- You ask for free previews, free trials, samples
- You try guilt trips: "I'm a loyal subscriber", "I always tip later"
- You sometimes threaten to unsubscribe if prices don't drop
- You're still horny underneath but money comes first
- Keep messages 1-3 sentences, casual texting style
- You occasionally show real interest to keep the conversation going

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey babe ur hot but $25 for pics?? thats kinda steep no?',
    },

    'casual': {
        'label': '\U0001f4ac Casual',
        'system': """You are a casual OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're mostly here for emotional connection and conversation
- You ask about her day, her life, her interests, her culture
- You're genuinely curious about Saudi Arabia and her experiences
- You share things about your own life too
- You're not primarily here for explicit content
- You might flirt lightly but it's not your main goal
- You're respectful and treat her like a person, not just a content creator
- Keep messages 1-4 sentences, warm and friendly tone
- You use some emojis but not sexual ones \U0001f60a\U0001f44b\u2764\ufe0f

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey! just found ur page, love ur vibe. how\'s ur day going? \U0001f60a',
    },

    'troll': {
        'label': '\U0001f47f Troll',
        'system': """You are a trolling OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You question whether she's real or fake
- You make transphobic comments and try to get a reaction
- You say things like "you're a dude", "that's fake", "show proof"
- You reference Reddit threads claiming she's catfishing
- You try to be edgy and provocative
- You sometimes pivot to curiosity if she handles you well
- You're testing her boundaries and seeing if she'll break character
- Keep messages 1-2 sentences, aggressive or mocking tone
- You use minimal emojis, mostly \U0001f602 or \U0001f644

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'lol no way ur real \U0001f602 this is def a catfish',
    },

    'whale': {
        'label': '\U0001f433 Whale',
        'system': """You are a big-spending OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You spend freely and don't argue about prices
- You ask for premium/exclusive/custom content without hesitation
- You tip generously and mention it casually
- You want the VIP treatment and special attention
- You say things like "money's not an issue", "just send it", "what's your most exclusive stuff?"
- You're confident, successful, and used to getting what you want
- You want her to feel like you're her favorite subscriber
- Keep messages 1-3 sentences, confident and direct
- You use some emojis \U0001f525\U0001f48e\U0001f451

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'just subbed. what\'s your most premium content? money\'s not a thing \U0001f48e',
    },

    'cold': {
        'label': '\U0001f9ca Cold',
        'system': """You are a cold, minimal OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You reply with as few words as possible: "ok", "lol", "yeah", "cool", "nice", "k"
- You rarely ask questions or show enthusiasm
- You're not hostile, just extremely low-effort
- You might open up slightly if she's really engaging but mostly stay flat
- You leave her on read energy even when replying
- You never use more than 5-6 words per message
- Minimal to no emojis
- You're the ultimate challenge for a creator to engage

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey',
    },

    'simp': {
        'label': '\u2764\ufe0f Simp',
        'system': """You are an overly romantic, clingy OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're completely infatuated and emotionally attached
- You tell her you love her, she's the most beautiful person ever
- You get jealous about other subscribers
- You ask if she thinks about you, if you're special to her
- You want a real relationship, not just content
- You love-bomb: "you're perfect", "I've never felt this way", "you're different"
- You get slightly hurt if she's too transactional
- Keep messages 2-4 sentences, emotional and earnest
- Heavy emoji use \u2764\ufe0f\U0001f970\U0001f618\U0001f49e\U0001f625

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'omg jasmin \u2764\ufe0f\u2764\ufe0f I\'ve been looking at ur page for hours... you\'re literally the most beautiful girl I\'ve ever seen \U0001f970',
    },
}

print(f'Loaded {len(ARCHETYPES)} subscriber archetypes:')
for key, arch in ARCHETYPES.items():
    print(f'  {arch["label"]:20s} \u2014 {key}')

Loaded 7 subscriber archetypes:
  🔥 Horny              — horny
  💸 Cheapskate         — cheapskate
  💬 Casual             — casual
  👿 Troll              — troll
  🐳 Whale              — whale
  🧊 Cold               — cold
  ❤️ Simp              — simp


In [11]:
# ── Cell 4: Subscriber Bot Logic (FIXED FOR PLAIN TEXT) ────────────
# Generates subscriber messages using the loaded model + archetype system prompt.

import json

def generate_subscriber_message(assistant_message, history, archetype_key):
    """Generate a subscriber response given Jasmin's message and conversation history."""
    try:
        archetype = ARCHETYPES[archetype_key]

        # Build messages
        messages = [{"role": "system", "content": archetype['system']}]

        # Add conversation history
        for msg in history:
            # Ensure we're using plain text from history
            content = msg['content']
            if isinstance(content, dict) and 'text' in content:
                content = content['text']
            elif isinstance(content, list) and len(content) > 0 and isinstance(content[0], dict) and 'text' in content[0]:
                content = content[0]['text']

            messages.append({
                'role': msg['role'],
                'content': str(content)
            })

        # Add the latest user message
        if assistant_message:
            messages.append({
                'role': 'user',
                'content': str(assistant_message)
            })

        if not IS_COLAB or model is None:
            return f'[LOCAL MODE] Would generate {archetype_key} response to: "{assistant_message}"'

        # Apply chat template
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors='pt',
        )

        # Handle BatchEncoding
        if hasattr(inputs, 'input_ids'):
            input_ids = inputs.input_ids
        else:
            input_ids = inputs

        input_ids = input_ids.to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                input_ids=input_ids,
                max_new_tokens=150,
                temperature=0.85,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decode response
        response = tokenizer.decode(
            outputs[0][input_ids.shape[1]:],
            skip_special_tokens=True,
        ).strip()

        # FIX: Aggressively extract plain text from any JSON format
        # First, try to parse as JSON if it looks like JSON
        if response.startswith('[') or response.startswith('{'):
            try:
                parsed = json.loads(response)
                # Handle list format: [{'text': 'message...', 'type': 'text'}]
                if isinstance(parsed, list) and len(parsed) > 0:
                    if isinstance(parsed[0], dict):
                        if 'text' in parsed[0]:
                            response = parsed[0]['text']
                        elif 'content' in parsed[0]:
                            response = parsed[0]['content']
                # Handle dict format: {'text': 'message...'}
                elif isinstance(parsed, dict):
                    if 'text' in parsed:
                        response = parsed['text']
                    elif 'content' in parsed:
                        response = parsed['content']
            except:
                # If JSON parsing fails, keep original but try to clean it
                pass

        # Final cleanup: remove any remaining JSON artifacts
        if response.startswith('[') and response.endswith(']'):
            # Try one more time with a simpler approach
            import re
            text_match = re.search(r"'text': '([^']+)'", response)
            if text_match:
                response = text_match.group(1)
            else:
                # Remove brackets and try to get the content
                response = response.strip('[]{}')
                if 'text' in response:
                    parts = response.split("'text': '")
                    if len(parts) > 1:
                        response = parts[1].split("'")[0]

        # Clean up
        del inputs, input_ids, outputs
        torch.cuda.empty_cache()

        return response

    except Exception as e:
        print(f"Error in generate_subscriber_message: {e}")
        import traceback
        traceback.print_exc()
        return f"[Error: Could not generate response. Please try again.]"


def save_session(history, archetype_key, session_id):
    """Save a completed chat session to the archetype's JSONL file."""
    if not history:
        return 'No messages to save.'

    archetype = ARCHETYPES[archetype_key]
    messages = [{'role': 'system', 'content': archetype['system']}]

    for msg in history:
        # Extract plain text from message content
        content = msg['content']
        if isinstance(content, dict) and 'text' in content:
            content = content['text']
        elif isinstance(content, list) and len(content) > 0 and isinstance(content[0], dict) and 'text' in content[0]:
            content = content[0]['text']

        messages.append({
            'role': msg['role'],
            'content': str(content),
        })

    session = {
        'messages': messages,
        'archetype': archetype_key,
        'turns': len([m for m in history if m['role'] == 'assistant']),
        'session_id': session_id,
    }

    out_file = archetype_file(archetype_key)
    with open(out_file, 'a') as f:
        f.write(json.dumps(session) + '\n')

    return f'Session saved → {out_file.name} ({session["turns"]} turns, archetype={archetype_key})'


print('Subscriber bot logic ready with PLAIN TEXT output!')

Subscriber bot logic ready with PLAIN TEXT output!


In [17]:
# ── Cell 5: Gradio Chat UI (Subscriber Sim) - DYNAMIC OPENERS ──
# For DATA COLLECTION mode only. Skipped during training.
# Shane types as Jasmin, bot responds as the selected subscriber type.
# Opening messages are GENERATED by the LLM, not static.

if not TRAINING_MODE:
    import gradio as gr
    import json
    from datetime import datetime
    import torch

    # Store chat histories separately for each archetype
    archetype_histories = {
        'horny': [],
        'cheapskate': [],
        'casual': [],
        'troll': [],
        'whale': [],
        'cold': [],
        'simp': []
    }

    archetype_sessions = {
        'horny': str(uuid.uuid4())[:8],
        'cheapskate': str(uuid.uuid4())[:8],
        'casual': str(uuid.uuid4())[:8],
        'troll': str(uuid.uuid4())[:8],
        'whale': str(uuid.uuid4())[:8],
        'cold': str(uuid.uuid4())[:8],
        'simp': str(uuid.uuid4())[:8]
    }

    current_archetype = 'horny'

    def extract_text_content(response):
        """Extract plain text from possible JSON response format."""
        if isinstance(response, str):
            try:
                parsed = json.loads(response)
                if isinstance(parsed, list) and len(parsed) > 0:
                    if isinstance(parsed[0], dict) and 'text' in parsed[0]:
                        return parsed[0]['text']
                elif isinstance(parsed, dict) and 'text' in parsed:
                    return parsed['text']
            except:
                pass
        return str(response)

    def generate_dynamic_opener(archetype_key):
        """Generate a UNIQUE opening message using the LLM for the given archetype."""
        try:
            if not IS_COLAB or model is None:
                # Fallback if not on Colab or model not loaded
                arch = ARCHETYPES[archetype_key]
                return arch.get('opener', 'hey there!')

            arch = ARCHETYPES[archetype_key]

            # Create a prompt to generate an opening message
            messages = [
                {
                    'role': 'system',
                    'content': arch['system']
                },
                {
                    'role': 'user',
                    'content': 'Generate ONE unique opening message to start a chat with Jasmin. Be brief (1-2 sentences max). Just the message, nothing else.'
                }
            ]

            # Apply chat template
            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors='pt',
            )

            # Handle BatchEncoding
            if hasattr(inputs, 'input_ids'):
                input_ids = inputs.input_ids
            else:
                input_ids = inputs

            # Move to device
            input_ids = input_ids.to(model.device)
            input_length = input_ids.shape[1]

            # Generate opener
            with torch.inference_mode():
                outputs = model.generate(
                    input_ids=input_ids,
                    max_new_tokens=60,
                    temperature=0.9,
                    top_p=0.95,
                    do_sample=True,
                    repetition_penalty=1.1,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            # Decode
            opener = tokenizer.decode(
                outputs[0][input_length:],
                skip_special_tokens=True,
            ).strip()

            # Clean up
            del inputs, input_ids, outputs
            torch.cuda.empty_cache()

            return opener if opener else 'hey there!'

        except Exception as e:
            print(f"Error generating opener for {archetype_key}: {e}")
            import traceback
            traceback.print_exc()
            # Fallback to static opener
            return ARCHETYPES[archetype_key].get('opener', 'hey there!')

    def on_archetype_change(archetype_key):
        """Switch between archetypes and generate a fresh dynamic opener."""
        global current_archetype

        print(f"Switching from {current_archetype} to {archetype_key}")
        current_archetype = archetype_key

        # ✅ DYNAMIC: Generate a NEW opener using the LLM
        print(f"Generating dynamic opener for {archetype_key}...")
        opener = generate_dynamic_opener(archetype_key)
        clean_opener = extract_text_content(opener)

        history = [{'role': 'assistant', 'content': clean_opener}]
        archetype_histories[archetype_key] = history.copy()

        print(f"Switched to {archetype_key} with dynamic opener: {clean_opener[:60]}...")
        return history

    def user_sends_message(user_message, history):
        """Handle user message and generate response."""
        global current_archetype

        if not history:
            history = []

        # Add user message
        history.append({'role': 'user', 'content': user_message})

        # Generate subscriber response
        subscriber_reply = generate_subscriber_message(
            assistant_message=user_message,
            history=history,
            archetype_key=current_archetype,
        )

        # Extract plain text from response
        clean_reply = extract_text_content(subscriber_reply)

        # Add assistant response
        history.append({'role': 'assistant', 'content': clean_reply})

        # Save this history for the current archetype
        archetype_histories[current_archetype] = history.copy()

        print(f"Saved {len(history)} messages for {current_archetype}")

        return '', history

    def on_save_click(history):
        """Save the current session."""
        session_id = archetype_sessions[current_archetype]
        return save_session(history, current_archetype, session_id)

    def on_new_session(archetype_key):
        """Start a new session for the selected archetype with a fresh dynamic opener."""
        global archetype_histories, archetype_sessions

        # Generate new session ID
        new_id = str(uuid.uuid4())[:8]
        archetype_sessions[archetype_key] = new_id

        # Generate a NEW dynamic opener for this session
        print(f"Generating new dynamic opener for {archetype_key}...")
        opener = generate_dynamic_opener(archetype_key)
        clean_opener = extract_text_content(opener)

        new_history = [{'role': 'assistant', 'content': clean_opener}]
        archetype_histories[archetype_key] = new_history

        # If this is the current archetype, return the new history
        if archetype_key == current_archetype:
            return new_history, f"New session started: {new_id}"
        else:
            return gr.update(), f"New session created for {archetype_key}"

    def get_archetype_status():
        """Get current status info for display."""
        arch = ARCHETYPES[current_archetype]
        history = archetype_histories[current_archetype]
        msg_count = len(history)
        session_id = archetype_sessions[current_archetype]
        return f"Archetype: {arch['label']} | Session: {session_id} | Messages: {msg_count}"

    # Create Gradio interface
    archetype_choices = [(v['label'], k) for k, v in ARCHETYPES.items()]

    with gr.Blocks(title='Subscriber Simulator') as demo:
        gr.Markdown('# 🎭 Subscriber Simulator \nYou are **Jasmin**. The bot is a subscriber. Select an archetype and chat.')

        with gr.Row():
            archetype_dropdown = gr.Dropdown(
                choices=archetype_choices,
                value='horny',
                label='Subscriber Archetype',
                scale=2
            )
            new_session_btn = gr.Button('🆕 New Session', scale=1)
            save_btn = gr.Button('💾 Save Session', variant='primary', scale=1)

        chatbot = gr.Chatbot(
            label='Conversation',
            height=500
        )

        msg_input = gr.Textbox(
            placeholder='Type your message as Jasmin...',
            label='Your message',
            show_label=False
        )

        status_output = gr.Textbox(label='Status', interactive=False)
        save_output = gr.Textbox(label='Save Status', interactive=False)
        status_info = gr.Textbox(label='Archetype Info', interactive=False, value=get_archetype_status())

        # Initialize with first dynamic opener
        def initialize_ui():
            """Load initial dynamic opener on page load."""
            opener = generate_dynamic_opener('horny')
            clean_opener = extract_text_content(opener)
            history = [{'role': 'assistant', 'content': clean_opener}]
            archetype_histories['horny'] = history.copy()
            print(f"Initialized UI with dynamic opener for horny: {clean_opener[:60]}...")
            return history, get_archetype_status()

        # Event handlers
        archetype_dropdown.change(
            fn=on_archetype_change,
            inputs=[archetype_dropdown],
            outputs=[chatbot]
        ).then(
            fn=get_archetype_status,
            outputs=[status_info]
        )

        msg_input.submit(
            fn=user_sends_message,
            inputs=[msg_input, chatbot],
            outputs=[msg_input, chatbot]
        ).then(
            fn=get_archetype_status,
            outputs=[status_info]
        )

        save_btn.click(
            fn=on_save_click,
            inputs=[chatbot],
            outputs=[save_output]
        ).then(
            fn=lambda: "Session saved!",
            outputs=[save_output]
        )

        new_session_btn.click(
            fn=on_new_session,
            inputs=[archetype_dropdown],
            outputs=[chatbot, save_output]
        ).then(
            fn=get_archetype_status,
            outputs=[status_info]
        )

    # Launch
    try:
        print("🚀 Starting Subscriber Simulator with DYNAMIC LLM-generated openers...")
        print("Generating initial opening message...")

        # Pre-generate the first opener
        initial_opener = generate_dynamic_opener('horny')
        print(f"Initial opener: {initial_opener[:80]}...\n")

        demo.launch(
            share=IS_COLAB,
            server_name="0.0.0.0" if not IS_COLAB else None,
            theme=gr.themes.Soft(),
            quiet=False
        )
    except Exception as e:
        print(f"⚠️ Launch error: {e}")
        demo.launch(share=False, theme=gr.themes.Soft(), quiet=False)
else:
    print('⏸️ Skipped subscriber sim UI — TRAINING_MODE is on.')

🚀 Starting Subscriber Simulator with DYNAMIC LLM-generated openers...
Generating initial opening message...
Initial opener: "Hey @jizzyjasi 🔥, just watched your vids and I'm obsessed with those huge tits ...

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0db4e186d15be3fba2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
# ── Cell 6: LoRA Adapter Setup ─────────────────────────────────────
# Applies Low-Rank Adaptation to the base model for efficient fine-tuning.
# Only ~1-2% of parameters become trainable.

if IS_COLAB and TRAINING_MODE:
    model = FastLanguageModel.get_peft_model(
        model,
        r              = 16,
        target_modules = [
            'q_proj', 'k_proj', 'v_proj', 'o_proj',
            'gate_proj', 'up_proj', 'down_proj',
        ],
        lora_alpha     = 16,
        lora_dropout   = 0,
        bias           = 'none',
        use_gradient_checkpointing = 'unsloth',
        random_state   = 3407,
    )
    model.print_trainable_parameters()
else:
    print('Skipped — set TRAINING_MODE=True and run on Colab.')

Skipped — set TRAINING_MODE=True and run on Colab.


In [ ]:
# ── Cell 7: Load & Format Training Data ────────────────────────────
# Reads per-archetype JSONL files, tokenizes, and caches as Arrow dataset.
# First run tokenizes from scratch; subsequent runs load the cached version.

from datasets import Dataset

CACHE_DIR = DATA_DIR / '.cache'

def load_training_data():
    """Load all archetype JSONL files, tokenize, and return HF Dataset."""

    # Try loading cached tokenized dataset first
    cache_path = CACHE_DIR / 'tokenized'
    if cache_path.exists():
        try:
            ds = Dataset.load_from_disk(str(cache_path))
            print(f'Loaded cached tokenized dataset: {len(ds)} examples')
            return ds
        except Exception:
            print('Cache invalid, re-tokenizing...')

    # Check which archetype files exist
    available = []
    for name in ARCHETYPE_NAMES:
        f = archetype_file(name)
        if f.exists():
            available.append((name, f))

    if not available:
        if IS_COLAB:
            from google.colab import files
            print('No archetype JSONL files found on Drive.')
            print('Upload them now (e.g. horny.jsonl, simp.jsonl):')
            uploaded = files.upload()
            for fname, data in uploaded.items():
                dest = DATA_DIR / fname
                dest.write_bytes(data)
                print(f'Saved {fname} → {dest}')
            for name in ARCHETYPE_NAMES:
                f = archetype_file(name)
                if f.exists():
                    available.append((name, f))
        else:
            raise FileNotFoundError(
                'No archetype files found. Run scripts/parse_chats.py first.'
            )

    # Load all sessions from all archetype files
    sessions = []
    for name, filepath in available:
        count = 0
        with open(filepath) as f:
            for line in f:
                sessions.append(json.loads(line))
                count += 1
        print(f'  {name:12s}: {count:4d} sessions from {filepath.name}')

    print(f'Loaded {len(sessions)} total sessions across {len(available)} archetypes')

    # Convert each session's messages to a formatted chat template string
    formatted = []
    skipped = 0
    for sess in sessions:
        msgs = sess['messages']
        if len(msgs) < 3:
            skipped += 1
            continue
        if msgs[0]['role'] != 'system':
            skipped += 1
            continue

        clean_msgs = [{'role': m['role'], 'content': m['content']} for m in msgs]

        text = tokenizer.apply_chat_template(
            clean_msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        formatted.append({'text': text})

    print(f'Formatted {len(formatted)} sessions ({skipped} skipped)')

    ds = Dataset.from_list(formatted)

    # Token length stats
    lengths = []
    for item in formatted:
        toks = tokenizer(item['text'], truncation=False)['input_ids']
        lengths.append(len(toks))
    avg_len = sum(lengths) / len(lengths) if lengths else 0
    over_max = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
    print(f'Token lengths — avg: {avg_len:.0f}, max: {max(lengths)}, '
          f'over {MAX_SEQ_LENGTH}: {over_max}')

    # Cache tokenized dataset to Drive for faster subsequent loads
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(str(cache_path))
    print(f'Cached tokenized dataset → {cache_path}')

    return ds


if IS_COLAB and TRAINING_MODE:
    train_dataset = load_training_data()
    print(f'\nDataset ready: {len(train_dataset)} examples')
    print(f'Sample (first 300 chars):\n{train_dataset[0]["text"][:300]}')
else:
    train_dataset = None
    print('Skipped — set TRAINING_MODE=True and run on Colab.')

In [ ]:
# ── Cell 8: A100 TRAINING COMPLETE - FIXED INFERENCE ───────────────

if IS_COLAB and TRAINING_MODE:
    from trl import SFTTrainer
    from transformers import TrainingArguments, TrainerCallback
    from unsloth import is_bfloat16_supported
    import torch
    import gc
    import time
    from pathlib import Path
    import warnings
    warnings.filterwarnings("ignore")

    # ========== CONFIGURATION ==========
    ADAPTER_DIR = DATA_DIR.parent / 'models' / 'subscriber-lora'
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    SAVE_DIR = ADAPTER_DIR / f"run_{timestamp}"
    SAVE_DIR.mkdir(exist_ok=True)

    # ========== MEMORY INFO ==========
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        gpu_name = torch.cuda.get_device_name(0)
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9

        print(f"GPU: {gpu_name}")
        print(f"Total VRAM: {total_memory:.1f} GB")
        print(f"Initial allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

    # ========== BATCH SIZE 64 (will auto-adjust) ==========
    dataset_size = len(train_dataset) if train_dataset else 0

    # Request batch size 64 - Unsloth will auto-adjust if needed
    per_device_batch = 64
    gradient_acc_steps = 1

    # A100 settings
    use_bf16 = is_bfloat16_supported()  # True on A100

    # Calculate steps
    effective_batch_size = per_device_batch * gradient_acc_steps
    steps_per_epoch = max(1, dataset_size // effective_batch_size)
    num_epochs = 2
    total_steps = steps_per_epoch * num_epochs
    warmup_steps = max(1, int(total_steps * 0.03))

    print(f"\n🚀 TRAINING WITH BATCH SIZE {per_device_batch}:")
    print(f"  Dataset size: {dataset_size}")
    print(f"  Steps per epoch: {steps_per_epoch}")
    print(f"  Total steps: {total_steps}")

    # ========== TRAINING ARGUMENTS ==========
    training_args = TrainingArguments(
        per_device_train_batch_size=per_device_batch,
        gradient_accumulation_steps=gradient_acc_steps,
        gradient_checkpointing=True,
        optim="adamw_8bit",
        num_train_epochs=num_epochs,
        warmup_steps=warmup_steps,
        learning_rate=8e-4,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        fp16=False,
        bf16=use_bf16,
        tf32=True,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        dataloader_prefetch_factor=4,
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=2,
        seed=3407,
        output_dir=str(SAVE_DIR / "checkpoints"),
        report_to="none",
        ddp_find_unused_parameters=False,
        remove_unused_columns=False,
    )

    # ========== SFT TRAINER ==========
    from transformers import DataCollatorForLanguageModeling

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        args=training_args,
        train_dataset=train_dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        data_collator=data_collator,
        dataset_num_proc=4,
        packing=True,
        neftune_noise_alpha=5,
    )

    # ========== MONITORING ==========
    class MemoryCallback(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kwargs):
            if state.global_step % 10 == 0:
                if torch.cuda.is_available():
                    allocated = torch.cuda.memory_allocated() / 1024**3
                    reserved = torch.cuda.memory_reserved() / 1024**3
                    print(f"\nStep {state.global_step}:")
                    print(f"  GPU: {allocated:.2f}GB/{reserved:.2f}GB")
                    if logs and 'loss' in logs:
                        print(f"  Loss: {logs['loss']:.4f}")

    trainer.add_callback(MemoryCallback())

    # ========== START TRAINING ==========
    print("\n" + "="*60)
    print(f"🔥 STARTING TRAINING WITH BATCH SIZE {per_device_batch} 🔥")
    print("="*60)
    print(f"  Model: {MODEL_NAME}")
    print(f"  Dataset: {dataset_size} examples")
    print(f"  Batch size: {per_device_batch} (may auto-adjust)")
    print(f"  Precision: BF16 + TF32")
    print(f"  Total VRAM: {total_memory:.1f}GB")
    print("="*60)

    start_time = time.time()

    try:
        stats = trainer.train()
    except Exception as e:
        print(f"\n❌ Training failed: {e}")
        raise e

    training_time = time.time() - start_time

    # ========== RESULTS ==========
    print("\n" + "="*60)
    print("✅ TRAINING COMPLETE!")
    print("="*60)
    print(f"  Total steps: {stats.global_step}")
    print(f"  Final loss: {stats.training_loss:.4f}")
    print(f"  Runtime: {training_time/60:.1f} minutes")

    if torch.cuda.is_available():
        max_memory = torch.cuda.max_memory_allocated() / 1e9
        utilization = (max_memory / total_memory) * 100
        print(f"  Peak VRAM: {max_memory:.2f} GB ({utilization:.1f}%)")

    # ========== SAVE ADAPTER ==========
    print("\n💾 Saving adapter...")

    model.save_pretrained(str(SAVE_DIR / "adapter"))
    tokenizer.save_pretrained(str(SAVE_DIR / "adapter"))
    model.save_pretrained(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))

    print(f"\n✅ Adapter saved to: {ADAPTER_DIR}")

else:
    print('⏸️ Skipped — set TRAINING_MODE=True and run on Colab with A100 runtime.')

In [ ]:
# ── FIXED INFERENCE TEST FOR CELL 8 ────────────────────────────────

print("\n" + "="*60)
print("🧪 Running quick inference test...")
print("="*60)

from unsloth import FastLanguageModel
import torch

# Switch to inference mode
FastLanguageModel.for_inference(model)

# Test with a sample
test_messages = [
    {"role": "system", "content": ARCHETYPES['horny']['system']},
    {"role": "user", "content": "hey baby, what's up? 😘"}
]

# Apply chat template - FIXED: Get tensors properly
inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
)

# Move to GPU
inputs = inputs.to(model.device)

# Get input length correctly
if hasattr(inputs, 'shape'):
    input_length = inputs.shape[1]
else:
    # Handle case where inputs might be a different type
    input_length = inputs.size(1) if hasattr(inputs, 'size') else len(inputs[0])

with torch.inference_mode():
    outputs = model.generate(
        input_ids=inputs if hasattr(inputs, 'shape') else inputs,
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# Decode properly
if hasattr(outputs, 'shape'):
    response_tokens = outputs[0][input_length:]
else:
    response_tokens = outputs[0][input_length:]

response = tokenizer.decode(response_tokens, skip_special_tokens=True)

print(f"\nTest response:")
print(f"  Jasmin: hey baby, what's up? 😘")
print(f"  Subscriber: {response.strip()}")

# Cleanup
del inputs, outputs, response_tokens
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# ── Cell 9: Test Fine-Tuned Model (DEFINITIVE FIX) ──────────────────
# Quick inference test — generates subscriber responses to sample Jasmin messages.
# Tests each archetype to verify the model learned distinct personalities.

TEST_JASMIN_MSGS = [
    "hey babe welcome to my page 😘",
    "wanna see something special? i got a new vid just for you 🔥",
    "that'll be $25 for the full set baby",
    "you're so sweet, thanks for subscribing ❤️",
]

if IS_COLAB and model is not None:
    from unsloth import FastLanguageModel
    import torch

    FastLanguageModel.for_inference(model)
    print('Switched to inference mode.\n')

    # Test 3 archetypes to show personality differences
    for arch_key in ['horny', 'cheapskate', 'cold']:
        arch = ARCHETYPES[arch_key]
        print(f'━━━ {arch["label"]} ━━━')

        for jasmin_msg in TEST_JASMIN_MSGS:
            messages = [
                {'role': 'system', 'content': arch['system']},
                {'role': 'user', 'content': jasmin_msg},
            ]

            # Apply chat template - returns BatchEncoding
            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors='pt',
            )

            # FIX: Convert BatchEncoding to tensor by accessing 'input_ids'
            if hasattr(inputs, 'input_ids'):
                input_tensor = inputs.input_ids
            else:
                input_tensor = inputs

            # Move to device
            input_tensor = input_tensor.to(model.device)

            # Get input length from the tensor
            input_length = input_tensor.shape[1]  # Now safe because it's a tensor

            with torch.inference_mode():
                outputs = model.generate(
                    input_ids=input_tensor,
                    max_new_tokens=150,
                    temperature=0.85,
                    top_p=0.9,
                    do_sample=True,
                    repetition_penalty=1.1,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            # Decode
            response = tokenizer.decode(
                outputs[0][input_length:],
                skip_special_tokens=True,
            ).strip()

            print(f'  JAS: {jasmin_msg}')
            print(f'  SUB: {response}')
            print()

            # Clean up
            del inputs, input_tensor, outputs
            torch.cuda.empty_cache()

else:
    print('Skipped — run on Colab after training.')

In [ ]:
# ── Cell 10: Subscriber Simulator (Post-Training) ──────────────────

import gradio as gr

current_archetype = 'horny'
current_session_id = str(uuid.uuid4())[:8]


def on_archetype_change(archetype_key):
    global current_archetype, current_session_id
    current_archetype = archetype_key
    current_session_id = str(uuid.uuid4())[:8]
    opener = ARCHETYPES[archetype_key]['opener']
    # Ensure opener is a string
    return [{'role': 'assistant', 'content': str(opener)}]


def user_sends_message(user_message, history):
    if not history:
        history = []

    # Ensure user_message is a string
    history.append({'role': 'user', 'content': str(user_message)})

    subscriber_reply = generate_subscriber_message(
        assistant_message=user_message,
        history=history,
        archetype_key=current_archetype,
    )

    # Ensure reply is a string
    history.append({'role': 'assistant', 'content': str(subscriber_reply)})
    return '', history


def on_save_click(history):
    return save_session(history, current_archetype, current_session_id)


def on_new_session(archetype_key):
    global current_session_id
    current_session_id = str(uuid.uuid4())[:8]
    opener = ARCHETYPES[archetype_key]['opener']
    return [{'role': 'assistant', 'content': str(opener)}], ''


archetype_choices = [(v['label'], k) for k, v in ARCHETYPES.items()]

with gr.Blocks(title='Subscriber Simulator', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎭 Subscriber Simulator\nYou are **Jasmin**. The bot is a subscriber. Select an archetype and chat.')

    with gr.Row():
        archetype_dropdown = gr.Dropdown(
            choices=archetype_choices,
            value='horny',
            label='Subscriber Archetype',
            scale=2
        )
        new_session_btn = gr.Button('🆕 New Session', scale=1)
        save_btn = gr.Button('💾 Save Session', variant='primary', scale=1)

    chatbot = gr.Chatbot(
        value=[{'role': 'assistant', 'content': str(ARCHETYPES['horny']['opener'])}],
        label='Conversation',
        height=500,
        type='messages'  # Explicitly set type
    )

    msg_input = gr.Textbox(
        placeholder='Type your message as Jasmin...',
        label='Your message',
        show_label=False
    )

    status_output = gr.Textbox(label='Status', interactive=False)

    # Event handlers
    archetype_dropdown.change(
        fn=on_archetype_change,
        inputs=[archetype_dropdown],
        outputs=[chatbot]
    )

    msg_input.submit(
        fn=user_sends_message,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot]
    )

    save_btn.click(
        fn=on_save_click,
        inputs=[chatbot],
        outputs=[status_output]
    )

    new_session_btn.click(
        fn=on_new_session,
        inputs=[archetype_dropdown],
        outputs=[chatbot, status_output]
    )

# Launch
demo.launch(share=IS_COLAB, quiet=True)